# Session 8 — Tabular Q-learning on Electronics

**Salih Tutun, PhD** · DAT 5668 · Module 4 · Practice · Explain → practice together

From Fall 2025 **Lab 4**. Session 7 = Monday fundamentals. Tonight = **Wednesday Q1–Q3** on Electronics. DQN → Session 9.

| Cue | Meaning |
| --- | --- |
| **RUN WITH ME** | Ready. Run. |
| **TRY THIS** | Fill `<<< FILL` / complete the function. |
| **CHANGE ME** | Change α / ε. **PAIR 20s**. |

![Q-table is decide](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-q-decide.png)
![Q-learning update](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/q-update.png)

*All diagrams load from GitHub raw URLs.*


# Part I — Topic

Session 7 (Lab Monday) taught Q-learning fundamentals on a 2-state toy. Tonight (Lab Wednesday path) we learn **Q(s,a)** from Electronics days — then harden the learner.

```text
Q(s,a) ← Q(s,a) + α [ r + γ max_a' Q(s',a') − Q(s,a) ]
```

ε-greedy: with probability ε pick random action; else `argmax Q(s,·)`.

### Two problems Session 7 already named

| Problem | What goes wrong on the desk | Tonight's fix |
| --- | --- | --- |
| **Overestimation** | One lucky high sample makes Order look better than it is | **Double Q** — two tables; pick with one, score with the other |
| **Correlated updates** | Day 100 always updates right after day 99 | **Experience replay** — store transitions; sample a batch |

**In one sentence each (memorize these):**

- **Double Q** = do not let the *same* table both choose the next action and grade that choice.
- **Experience replay** = save past `(s, a, r, s')` days and learn from a *random mix*, not only from “tomorrow after today.”

**Flow:** toy 3-state (Lab Q1) → real Electronics states (Lab Q2) → you write the update → **Double Q** → **experience replay** → optional 10 bins (Lab Q3). DQN (neural net) → Session 9.

**JUMP BACK** until I say OPEN.


---
# Part II — Practice


---
## Timed practice tonight (instructor clock)

| Block | Time | Stop | Done when |
| --- | --- | --- | --- |
| **Practice 1** | **10 min** | 4b Double Q | **CHECK 4b** + fill **REPORT 1** (which Q better / less inflated) |
| **Practice 2** | **10 min** | 4c Experience replay | **CHECK 4c** + fill **REPORT 2** (what replay reused) |

**Before these two blocks:** finish Stops 0–4 until **CHECK 4 — PASS**.  
**After:** Stop 6–7 deploy talk + raise a hand. Stop 5 bins = only if time left.


## Stop 0 — Imports


In [ ]:
# STOP 0a — RUN WITH ME · imports

from pathlib import Path
from collections import Counter
import csv
import random
import statistics

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
PRODUCT = "Electronics"
RNG = random.Random(7)


### 0b — course_data (RUN WITH ME)


In [ ]:
# STOP 0b — RUN WITH ME · course_data

def course_data(name: str) -> Path:
    # Prefer Drive, then local folders, then GitHub download (Colab-safe)
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path
    if name in {"inventory_dataset_full.csv", "inventory_dataset.csv"}:
        import urllib.request
        dest = Path("/content/data") if Path("/content").exists() else Path("data")
        dest.mkdir(parents=True, exist_ok=True)
        out = dest / name
        url = "https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
        print("Downloading", name, "from GitHub…")
        urllib.request.urlretrieve(url, out)
        return out
    raise FileNotFoundError("Could not find " + name + ". Mount Drive DAT5668/data or use GitHub fallback.")


## Stop 1 — Load Electronics


In [ ]:
# STOP 1 — RUN WITH ME · load

def load_electronics_desk():
    rows = []
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product") or "").strip() != PRODUCT:
                continue
            rows.append({
                "Date": row["Date"],
                "Current_Inventory": int(float(row["Current_Inventory"])),
                "Demand": int(float(row["Demand"])),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),
                "Profit": float(row.get("Profit") or 0),
            })
    if not rows:
        raise ValueError("No Electronics rows")
    rows.sort(key=lambda r: r["Date"])
    return rows


ROWS = load_electronics_desk()
N_DAYS = len(ROWS)
AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)
print("Electronics days:", N_DAYS, "| avg demand:", round(AVG_DEMAND, 1))
print("Historical actions:", dict(Counter(r["Action_Taken"] for r in ROWS)))
print("CHECK load —", "PASS" if N_DAYS == 730 else "NOT YET")


## Stop 2 — Lab 4 Q1 warm-up (toy 3 states)

Hand-crafted rewards. Random next state. See the Q-update move before we wire the real log.


In [ ]:
# STOP 2a — RUN WITH ME · toy Q-learning
# Tiny 3-state world so you can *see* one Q-cell move before Electronics.

STATES = ["Low Inventory", "Medium Inventory", "High Inventory"]
ACTIONS = ["Order", "Maintain", "Reduce"]
# Hand-crafted rewards: clearly want Order when Low, Reduce when High
REWARDS = {
    ("Low Inventory", "Order"): 10, ("Low Inventory", "Maintain"): -5, ("Low Inventory", "Reduce"): -10,
    ("Medium Inventory", "Order"): 2, ("Medium Inventory", "Maintain"): 5, ("Medium Inventory", "Reduce"): 2,
    ("High Inventory", "Order"): -10, ("High Inventory", "Maintain"): -5, ("High Inventory", "Reduce"): 10,
}


def train_toy(alpha=0.1, epsilon=0.1, gamma=0.9, episodes=100):
    # Start with an empty table: every Q(s,a) = 0
    Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
    tracked = []  # values of Q(Low, Order) after each update of that cell
    for _ in range(episodes):
        state = RNG.choice(STATES)  # start somewhere at random
        for _step in range(10):     # short trajectory
            # ε-greedy: explore or pick argmax Q[state]
            action = RNG.choice(ACTIONS) if RNG.random() < epsilon else max(Q[state], key=Q[state].get)
            reward = REWARDS[(state, action)]  # look up hand-crafted reward
            nxt = RNG.choice(STATES)           # toy next state = random (not the CSV)
            old = Q[state][action]
            # One TD update: move toward r + γ max Q[next]
            Q[state][action] = old + alpha * (reward + gamma * max(Q[nxt].values()) - old)
            if state == "Low Inventory" and action == "Order":
                tracked.append(Q[state][action])  # watch this cell climb
            state = nxt  # continue from next state
    policy = {s: max(Q[s], key=Q[s].get) for s in STATES}  # greedy policy
    return Q, policy, tracked


Q_toy, policy_toy, tracked_toy = train_toy()
print("Toy policy:", policy_toy)
print("CHECK toy —", "PASS" if policy_toy["Low Inventory"] == "Order" and policy_toy["High Inventory"] == "Reduce" else "LOOK")

if plt is not None and tracked_toy:
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.plot(tracked_toy, color="#2F5D50")
    ax.set_title("Toy Q(Low, Order)")
    fig.tight_layout()
    plt.show()


### 2b — CHANGE ME · α / ε on the toy


In [ ]:
# STOP 2b — CHANGE ME

ALPHA = 0.1    # <<< CHANGE ME: 0.05 / 0.3 / 1.0
EPSILON = 0.1  # <<< CHANGE ME: 0.0 / 0.5

_, policy2, tracked2 = train_toy(alpha=ALPHA, epsilon=EPSILON)
print("alpha", ALPHA, "epsilon", EPSILON, "→", policy2)
if plt is not None and tracked2:
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.plot(tracked2, color="#C45C26")
    ax.set_title(f"Q(Low,Order) α={ALPHA} ε={EPSILON}")
    fig.tight_layout()
    plt.show()


## Stop 3 — Lab 4 Q2: states from real Electronics days

`Current_Inventory / avg Demand` → Low / Medium / High (same idea as Session 6).


In [ ]:
# STOP 3 — RUN WITH ME · categorize
# Turn numeric inventory into the same Low / Medium / High labels Q-learning uses.


def categorize_inventory(inv, avg_demand, low_cut=0.5, high_cut=2.0):
    # ratio = how many "average demand days" of stock we hold
    ratio = float(inv) / max(avg_demand, 1e-9)  # avoid divide-by-zero
    if ratio < low_cut:
        return "Low Inventory"       # thin shelf
    if ratio <= high_cut:
        return "Medium Inventory"    # comfortable band
    return "High Inventory"          # overstocked


# Label every Electronics day once (used by Stops 4 / 4b / 4c)
for r in ROWS:
    r["State"] = categorize_inventory(r["Current_Inventory"], AVG_DEMAND)

counts = Counter(r["State"] for r in ROWS)  # how many days in each band
print("State mix:", dict(counts))
if plt is not None:
    labels = ["Low Inventory", "Medium Inventory", "High Inventory"]
    fig, ax = plt.subplots(figsize=(5, 2.6))
    ax.bar(labels, [counts[k] for k in labels], color=["#C45C26", "#2F5D50", "#1F4E79"])
    ax.set_title("Electronics state mix")
    fig.tight_layout()
    plt.show()


## Stop 4 — YOU write ε-greedy + Q-update (Lab 4 core)

![Update](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/q-update.png)

Walk consecutive Electronics days. Complete the two functions. Re-run until **CHECK 4 — PASS**.


In [ ]:
# STOP 4 — TRY THIS · Q-learning on Electronics days
# Goal: walk real days, pick actions with ε-greedy, update one Q-table.

# --- environment pieces (same idea as Lab 4) ---
STATES = ["Low Inventory", "Medium Inventory", "High Inventory"]  # discrete shelf bands
ACTIONS = ["Order", "Maintain", "Reduce"]  # what the decide box can choose
# Rule-table rewards (NOT the Profit column). Designed so Low→Order, Med→Maintain, High→Reduce.
REWARDS_DATA = {
    ("Low Inventory", "Order"): 2, ("Low Inventory", "Maintain"): -1, ("Low Inventory", "Reduce"): -2,
    ("Medium Inventory", "Order"): 0, ("Medium Inventory", "Maintain"): 1, ("Medium Inventory", "Reduce"): 0,
    ("High Inventory", "Order"): -2, ("High Inventory", "Maintain"): -1, ("High Inventory", "Reduce"): 2,
}

alpha = 0.3          # learning rate: how hard we pull Q toward the new target
epsilon = 0.5        # <<< CHANGE ME after first PASS — exploration probability
gamma = 0.7          # discount: how much we care about tomorrow's max Q
n_episodes = 50      # how many times we walk the whole Electronics log

# Q[state][action] starts at 0 — "I don't know yet"
Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
tracked = []  # store Q(Low, Order) after each update so we can plot learning


def epsilon_greedy(state, Q, epsilon):
    # <<< FILL: if random < epsilon → random action; else argmax Q[state]
    # Explore = try something new. Exploit = pick the action with highest Q so far.
    return None  # replace this line


def q_update(Q, state, action, reward, next_state, alpha, gamma):
    # <<< FILL: Q[s,a] ← Q[s,a] + α (r + γ max Q[s'] − Q[s,a])
    # target = reward now + discounted best guess for next state
    # TD error = target − old Q; we move a fraction α of the way toward the target
    return None  # replace: update Q in place


ready = True
try:
    for _ in range(n_episodes):                      # outer loop: repeat over the year many times
        for i in range(len(ROWS) - 1):               # inner loop: consecutive days in the log
            state = ROWS[i]["State"]                 # today's band (from Stop 3)
            action = epsilon_greedy(state, Q, epsilon)  # decide box (ε-greedy)
            if action not in ACTIONS:
                raise ValueError("epsilon_greedy must return an action string")
            reward = REWARDS_DATA[(state, action)]   # look up rule-table reward for (s,a)
            next_state = ROWS[i + 1]["State"]        # tomorrow's band from the FILE (action does not move stock)
            before = Q[state][action]                # remember old Q so we can see if it moved
            q_update(Q, state, action, reward, next_state, alpha, gamma)  # one TD update
            if Q[state][action] == before and alpha != 0:
                pass  # early zeros can look flat; keep going
            if state == "Low Inventory" and action == "Order":
                tracked.append(Q[state][action])     # track one cell for the learning curve
except Exception as err:
    ready = False
    print("NOT YET —", err)
    print("Fill epsilon_greedy and q_update, then re-run this cell.")

if ready:
    # Policy = greedy action in each state after learning
    policy = {s: max(Q[s], key=Q[s].get) for s in STATES}
    print("Policy:", policy)
    ok = (
        policy["Low Inventory"] == "Order"
        and policy["Medium Inventory"] == "Maintain"
        and policy["High Inventory"] == "Reduce"
    )
    print("CHECK 4 —", "PASS" if ok else "LOOK — expected Low→Order, Med→Maintain, High→Reduce")
    # How often would this policy have matched the historical Action_Taken?
    agree = 0
    for r in ROWS:
        if policy[r["State"]] == r["Action_Taken"]:
            agree += 1
    print("Agree with historical Action_Taken:", agree, "/", N_DAYS, f"({agree/N_DAYS:.1%})")
    if plt is not None:
        fig, axes = plt.subplots(1, 2, figsize=(8.5, 2.8))
        if tracked:
            axes[0].plot(tracked[:: max(1, len(tracked)//200)], color="#1F4E79")
        axes[0].set_title("Q(Low, Order) updates")
        axes[1].bar(["agree", "disagree"], [agree, N_DAYS - agree], color=["#2F5D50", "#8B3A2A"])
        axes[1].set_title("Policy vs history")
        fig.tight_layout()
        plt.show()


### Hint (after you try)

```python
def epsilon_greedy(state, Q, epsilon):
    if RNG.random() < epsilon:
        return RNG.choice(ACTIONS)
    return max(Q[state], key=Q[state].get)

def q_update(Q, state, action, reward, next_state, alpha, gamma):
    old = Q[state][action]
    Q[state][action] = old + alpha * (reward + gamma * max(Q[next_state].values()) - old)
```


## Stop 4b — What is Double Q-learning? (explain → TRY THIS)

> **Practice 1 · ⏱ 10 min** — Run TOY 4b with me (1–2 min). Then fill only `double_q_update`. Raise a hand on **CHECK 4b — PASS**, then fill **REPORT 1** and share one number.

### Plain definition

**Double Q-learning** is Q-learning with **two** value tables, `QA` and `QB`, instead of one.

On each update we:

1. Flip a coin to choose which table to **update**.
2. Use that table to **pick** the best next action `a* = argmax`.
3. Use the **other** table to **score** how good `a*` is.

So the chooser and the grader are never the same notebook.

### Why we need it (overestimation)

In Stop 4 the standard target is:

```text
target = r + γ · max_a' Q(s', a')
```

The **same** `Q` both:

1. **picks** the best next action (`argmax`), and
2. **scores** that action (`Q[s', a*]`).

If noise or a lucky day inflated one entry, `argmax` keeps choosing it and the update pushes it higher. Session 7 called this **overestimation**.

**Inventory story:** after three lucky restock days, `Q(Low, Order)` looks huge. Standard Q keeps picking Order *and* scoring Order with that same optimistic number. The shelf policy can get stuck “always order” even when Maintain would be fine.

### Desk analogy

Think of two managers reviewing the same Electronics log:

| Manager | Job tonight |
| --- | --- |
| `QA` | Proposes “next best action” |
| `QB` | Grades that proposal with *its* numbers (or the reverse after the coin flip) |

Neither manager grades their own homework. That is Double Q.

### The update (write this on the board)

| Coin | Update this table | Pick next action with | Score that pick with |
| --- | --- | --- | --- |
| Heads | `QA` | `argmax QA(s', ·)` → `a*` | `QB(s', a*)` |
| Tails | `QB` | `argmax QB(s', ·)` → `a*` | `QA(s', a*)` |

```text
update QA:  a* = argmax QA(s', ·)     then  QA ← QA + α [ r + γ QB(s', a*) − QA ]
update QB:  a* = argmax QB(s', ·)     then  QB ← QB + α [ r + γ QA(s', a*) − QB ]
```

### Tiny numeric sketch (one step)

Suppose next state is Medium, and:

```text
QA[Medium] = {Order: 5.0, Maintain: 4.0, Reduce: 1.0}
QB[Medium] = {Order: 2.0, Maintain: 3.5, Reduce: 1.0}
```

If we update **QA** (heads): pick `a* = Order` from QA (5.0 wins), but score with **QB**: `QB[Medium, Order] = 2.0` — not 5.0.

Standard Q would have used 5.0 in the target. Double Q uses 2.0. That is the bias check.

### How we act and how we judge success

- **Act (ε-greedy):** average `(QA + QB) / 2` for `argmax` (still explore with probability ε).
- **Policy we report:** `arg max_a (QA(s,a) + QB(s,a)) / 2`.
- **Check:** same as Stop 4 — Low→Order, Medium→Maintain, High→Reduce (**CHECK 4b — PASS**).
- **Compare:** print `Q(Low, Order)` from Stop 4 vs the Double average. Ask: *which looks more optimistic?*

**Before the blank:** run **TOY 4b** (next cells) so the room sees max vs pick/score.

Fill `double_q_update` below. Re-run until CHECK 4b passes.


### TOY 4b — See overestimation in 20 seconds (RUN WITH ME)

One fake next state. Ten noisy action values. True mean of every action is **0**.

- **Standard Q** takes `max` of those noises → the target is too high (optimistic).
- **Double Q** picks `a*` from list A, scores it with list B → closer to 0.

Run the next cell. Watch the printed numbers and the bar chart.


In [ ]:
# TOY 4b — RUN WITH ME · why max overestimates (no Electronics yet)
# Tiny demo: true value of every action is 0; we only see noisy estimates.

# Fake Q-row for one next state. True value of every action is 0.
# We only see noisy estimates (lucky / unlucky samples).
noise_A = [0.4, -0.2, 1.8, -0.5, 0.1, 0.9, -1.1, 0.3, -0.7, 0.2]  # pick list (table A)
noise_B = [-0.3, 0.5, 0.2, -0.1, 0.0, -0.4, 0.6, -0.2, 0.1, -0.8]  # score list (table B)
actions = [f"a{i}" for i in range(len(noise_A))]  # fake action names a0..a9

# --- Standard Q: pick AND score with the SAME list ---
a_star_std = max(range(len(noise_A)), key=lambda i: noise_A[i])  # index of the max noise
target_std = noise_A[a_star_std]  # score = that same max → optimistic if noise was lucky

# --- Double Q idea: pick with A, score with B ---
a_star_d = max(range(len(noise_A)), key=lambda i: noise_A[i])  # still pick from A
target_d = noise_B[a_star_d]  # but grade that pick using B (usually closer to 0)

print("True mean of every action: 0")
print(f"Standard Q  picks {actions[a_star_std]}  scores with SAME list → target = {target_std:.2f}")
print(f"Double Q    picks {actions[a_star_d]}  scores with OTHER list → target = {target_d:.2f}")
print("CHECK toy 4b —", "PASS" if target_std > target_d else "LOOK (std should look more optimistic)")

if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 2.8))
    ax.bar(actions, noise_A, color="#1F4E79", alpha=0.85, label="list A (pick)")
    ax.axhline(0, color="black", linewidth=0.8)  # true mean
    ax.scatter([a_star_std], [target_std], s=80, color="#C45C26", zorder=3, label=f"std max={target_std:.1f}")
    ax.scatter([a_star_d], [target_d], s=80, color="#6B4C9A", zorder=3, label=f"double score={target_d:.1f}")
    ax.set_title("Same noises: max (orange) vs pick-A/score-B (purple)")
    ax.legend(fontsize=8, loc="upper right")
    fig.tight_layout()
    plt.show()


In [ ]:
# STOP 4b — TRY THIS · Double Q on Electronics days
# Goal: two tables QA, QB — pick with one, score with the other (fixes overestimation).

QA = {s: {a: 0.0 for a in ACTIONS} for s in STATES}  # first Q-table
QB = {s: {a: 0.0 for a in ACTIONS} for s in STATES}  # second Q-table
tracked_double = []  # average Q(Low, Order) for the plot


def combined_q(state):
    # Act using the average of both tables (more stable than either alone)
    return {a: (QA[state][a] + QB[state][a]) / 2.0 for a in ACTIONS}


def epsilon_greedy_double(state, epsilon):
    # Same ε-greedy idea as Stop 4, but greediness uses combined_q
    if RNG.random() < epsilon:
        return RNG.choice(ACTIONS)  # explore
    cq = combined_q(state)
    return max(cq, key=cq.get)  # exploit average Q


def double_q_update(state, action, reward, next_state, alpha, gamma):
    # <<< FILL: coin flip — update one table, pick a* with that table, score with the other
    # if update QA: a_star = argmax QA[next]; QA[s,a] ← QA[s,a] + α (r + γ QB[next][a_star] − QA[s,a])
    # else:         a_star = argmax QB[next]; QB[s,a] ← QB[s,a] + α (r + γ QA[next][a_star] − QB[s,a])
    # Key idea: the table that PICKS a* is never the table that SCORES a*.
    return None  # replace: update QA or QB in place


ready_d = True
try:
    for _ in range(n_episodes):                      # reuse α, ε, γ, n_episodes from Stop 4
        for i in range(len(ROWS) - 1):
            state = ROWS[i]["State"]                 # today
            action = epsilon_greedy_double(state, epsilon)  # decide with average Q
            if action not in ACTIONS:
                raise ValueError("epsilon_greedy_double must return an action string")
            reward = REWARDS_DATA[(state, action)]   # same rule rewards as Stop 4
            next_state = ROWS[i + 1]["State"]        # tomorrow from the file
            before = (QA[state][action], QB[state][action])  # snapshot both tables
            double_q_update(state, action, reward, next_state, alpha, gamma)
            after = (QA[state][action], QB[state][action])
            if after == before and alpha != 0:
                pass  # early zeros can look flat; keep going
            if state == "Low Inventory" and action == "Order":
                # track the AVERAGE of the two tables for this cell
                tracked_double.append((QA[state][action] + QB[state][action]) / 2.0)
except Exception as err:
    ready_d = False
    print("NOT YET —", err)
    print("Fill double_q_update, then re-run this cell.")

if ready_d:
    # Policy from average Q (not from QA or QB alone)
    policy_d = {s: max(combined_q(s), key=combined_q(s).get) for s in STATES}
    print("Double-Q policy:", policy_d)
    ok_d = (
        policy_d["Low Inventory"] == "Order"
        and policy_d["Medium Inventory"] == "Maintain"
        and policy_d["High Inventory"] == "Reduce"
    )
    print("CHECK 4b —", "PASS" if ok_d else "LOOK — expected Low→Order, Med→Maintain, High→Reduce")
    # Compare optimism: single Q from Stop 4 vs average of QA+QB
    if "Q" in globals():
        print("Compare Q(Low, Order): single={:.2f}  double-avg={:.2f}".format(
            Q["Low Inventory"]["Order"],
            (QA["Low Inventory"]["Order"] + QB["Low Inventory"]["Order"]) / 2.0,
        ))
        print("PAIR 20s: which number looks more optimistic? Why?")
        print("# === SEE (for REPORT 1) ===")
        print("Better result to report: Double Q keeps the same policy with a less inflated Q(Low, Order).")
    if plt is not None and tracked_double:
        fig, ax = plt.subplots(figsize=(5, 2.6))
        step = max(1, len(tracked_double) // 200)
        ax.plot(tracked_double[::step], color="#6B4C9A", label="double avg")
        if "tracked" in globals() and tracked:
            ax.plot(tracked[:: max(1, len(tracked) // 200)], color="#1F4E79", alpha=0.7, label="single Q")
        ax.set_title("Q(Low, Order): single vs Double Q")
        ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()


### Hint 4b (after you try)

```python
def double_q_update(state, action, reward, next_state, alpha, gamma):
    if RNG.random() < 0.5:
        a_star = max(QA[next_state], key=QA[next_state].get)
        old = QA[state][action]
        QA[state][action] = old + alpha * (reward + gamma * QB[next_state][a_star] - old)
    else:
        a_star = max(QB[next_state], key=QB[next_state].get)
        old = QB[state][action]
        QB[state][action] = old + alpha * (reward + gamma * QA[next_state][a_star] - old)
```


### After CHECK 4b — what you should be able to say

**What is Double Q?** Two Q-tables; one picks the next action, the other scores it (coin flip chooses which role).

1. Standard Q uses one table to **pick** and **score** → can **overestimate**.
2. Double Q: pick with `QA`, score with `QB` (or the reverse).
3. We still want the same Electronics policy; the win is a **less inflated** Q.

If single `Q(Low, Order)` is higher than the Double average, that is the bias showing up — not a bug in CHECK 4b.


### Practice 1 REPORT · try → see → tell the room (⏱ end of 10 min)

After **CHECK 4b — PASS**, look at the printed comparison and the plot. Fill the report cell. Be ready to say one number out loud.


In [ ]:
# PRACTICE 1 REPORT — TRY THIS · Double Q

# Use numbers printed above (Stop 4 vs Stop 4b). Replace the blanks.

practice1_single_Q_low_order = None   # <<< FILL: float from "single=..."
practice1_double_avg_low_order = None  # <<< FILL: float from "double-avg=..."
practice1_more_optimistic = ""         # <<< FILL: "single" or "double"
practice1_policy_still_good = None     # <<< FILL: True if CHECK 4b PASS
practice1_what_got_better = ""         # <<< FILL: one sentence (bias / optimism / pick vs score)

print("=== PRACTICE 1 REPORT ===")
print("single Q(Low,Order):", practice1_single_Q_low_order)
print("double avg Q(Low,Order):", practice1_double_avg_low_order)
print("more optimistic:", practice1_more_optimistic)
print("CHECK 4b still good policy:", practice1_policy_still_good)
print("What got better:", practice1_what_got_better)

ok = (
    practice1_single_Q_low_order is not None
    and practice1_double_avg_low_order is not None
    and practice1_more_optimistic in {"single", "double"}
    and practice1_policy_still_good is True
    and len(practice1_what_got_better.strip()) > 10
)
print("REPORT 1 —", "PASS (ready to share)" if ok else "NOT YET — fill every blank")
if ok and practice1_single_Q_low_order > practice1_double_avg_low_order:
    print("Nice: single looks more optimistic — that is the overestimation Double Q checks.")


## Stop 4c — What is experience replay? (explain → TRY THIS · stretch)

> **Practice 2 · ⏱ 10 min** — Run TOY 4c with me (1–2 min). Then fill `sample_batch` + buffer `append`. Raise a hand on **CHECK 4c — PASS**, then fill **REPORT 2**. If stuck: still write the one-sentence *what got better*.

### Plain definition

**Experience replay** means: do **not** learn only from the transition you just saw.

Instead:

1. **Store** every transition `(s, a, r, s')` in a memory buffer (a list / `deque`).
2. **Sample** a mini-batch of past transitions at random.
3. **Update** Q (tonight: Double Q) on that batch.

The agent still *acts* on today’s Electronics day. It *learns* from a shuffled mix of many past days.

### Why walking the CSV is a weak training diet

Stop 4 and 4b update on **consecutive** Electronics days:

```text
day i → (s, a, r, s') → day i+1 → (s', a', r', s'') → …
```

Every update is glued to tomorrow. That creates three problems:

| Issue | Desk meaning |
| --- | --- |
| **Correlation** | March 10 and March 11 are almost the same story — the learner sees one long streak, not independent samples |
| **Rare days forgotten** | A stockout week is used once, then buried — no reuse |
| **Order matters too much** | Shuffling *when* we learn from a day should not rewrite the whole table |

### Desk analogy

Without replay: a trainee shadows the store **in calendar order** only — Monday, then Tuesday, then Wednesday. They overfit the sequence.

With replay: after each day they also pull **32 random past days** from a folder of notes and practice those again. Rare stockouts get revisited. Neighboring days are no longer glued together in every update.

### The loop (say it out loud)

```text
act on day i
  → append (s, a, r, s') to buffer
  → sample a batch of past transitions
  → Double-Q update on each sample in the batch
```

Say it: *break the day-to-day correlation; reuse past Electronics days.*

### What each piece is

| Piece | Job |
| --- | --- |
| `replay_buffer` | Memory of past transitions (max length = buffer size; oldest drop out when full) |
| `(s, a, r, s')` | One experience: state, action taken, reward, next state |
| `sample_batch` | Random sample of up to `BATCH_SIZE` experiences |
| Double Q on the batch | Same pick/score split as Stop 4b, but on sampled days |

### What experience replay is *not*

- It is **not** a neural network. Tonight the Q values are still tables.
- It is **not** changing the reward rules. Same `REWARDS_DATA`.
- It is **not** “predicting demand.” It only reorders *when* we learn from logged days.

### Bridge to Session 9 (do not code a network tonight)

| Tonight (tabular) | Session 9 (DQN) |
| --- | --- |
| Two Q-**tables** | Q-**networks** |
| Same experience buffer | Same idea: store and sample transitions |
| Rule-table rewards | Same loop; representation changes |

**Before the blank:** run **TOY 4c** (next cells) so the room sees calendar order vs a random batch.

Fill `sample_batch` and the `append` line. Re-run until **CHECK 4c — PASS**.


### TOY 4c — See a replay batch in 20 seconds (RUN WITH ME)

Five fake store days in **calendar order**. Without replay you always learn 1→2→3→4→5.

With replay you still *live* day 5, but you *learn* from a **random batch** of past days (and can revisit a rare stockout).

Run the next cell. Compare the two printouts.


In [ ]:
# TOY 4c — RUN WITH ME · sequential days vs a replay batch
# Tiny demo: calendar order vs learning from a random mix of past days.

# Tiny fake log (not the CSV). One rare-ish miss buried on Wednesday.
toy_days = [
    ("Mon", "Low", "Order", +2),
    ("Tue", "Low", "Order", +2),
    ("Wed", "Low", "Maintain", -1),   # rare-ish miss — we want this reused
    ("Thu", "Medium", "Maintain", +1),
    ("Fri", "High", "Reduce", +2),
]

print("Calendar order (no replay) — every update glued to tomorrow:")
for i, (day, s, a, r) in enumerate(toy_days):
    nxt = toy_days[i + 1][1] if i + 1 < len(toy_days) else "END"
    print(f"  learn from {day}: ({s}, {a}, r={r:+d}, next={nxt})")

# Live through the week once: store every transition (s, a, r, s')
buffer = []
for i in range(len(toy_days) - 1):
    day, s, a, r = toy_days[i]
    sp = toy_days[i + 1][1]  # next state's label
    buffer.append((day, s, a, r, sp))  # one experience in memory

# Experience replay: sample a mini-batch at random (not in calendar order)
batch = RNG.sample(buffer, k=min(3, len(buffer)))
print("\nOne experience-replay batch (same memories, shuffled):")
for day, s, a, r, sp in batch:
    print(f"  reuse {day}: ({s}, {a}, r={r:+d}, next={sp})")

# Show reuse: sample many times — how often does Wed (the miss) reappear?
hits = 0
draws = 20
for _ in range(draws):
    b = RNG.sample(buffer, k=min(3, len(buffer)))
    if any(t[0] == "Wed" for t in b):
        hits += 1
print(f"\nWed (Maintain miss) appeared in {hits}/{draws} random batches — rare days get reused.")
print("CHECK toy 4c —", "PASS" if len(batch) == 3 and len(buffer) == 4 else "LOOK")


In [ ]:
# STOP 4c — TRY THIS · Double Q + experience replay
# Goal: store each day, sample a batch, Double-Q update on past days (not only tomorrow).

from collections import deque  # fixed-size queue: oldest drops when full

BUFFER_SIZE = 2000       # max transitions kept in memory
BATCH_SIZE = 32          # how many past days we learn from each step
REPLAY_EPISODES = 30     # shorter: each step also samples a batch

replay_buffer = deque(maxlen=BUFFER_SIZE)  # experience memory
QA_r = {s: {a: 0.0 for a in ACTIONS} for s in STATES}  # Double Q table A (replay run)
QB_r = {s: {a: 0.0 for a in ACTIONS} for s in STATES}  # Double Q table B (replay run)
episode_returns = []  # sum of rule rewards per episode (for the plot)


def combined_q_r(state):
    # Average of both tables — used for ε-greedy acting
    return {a: (QA_r[state][a] + QB_r[state][a]) / 2.0 for a in ACTIONS}


def sample_batch(buffer, batch_size):
    # <<< FILL: return a random sample of up to batch_size transitions from buffer
    # hint: RNG.sample(list(buffer), k)  with k = min(len(buffer), batch_size)
    # Empty buffer → return [] so training can start filling memory first
    return []  # replace this line


def double_q_update_tables(QA_tab, QB_tab, state, action, reward, next_state, alpha, gamma):
    # Same Double-Q coin flip as Stop 4b, but on whatever tables we pass in
    if RNG.random() < 0.5:
        a_star = max(QA_tab[next_state], key=QA_tab[next_state].get)  # pick with A
        old = QA_tab[state][action]
        QA_tab[state][action] = old + alpha * (reward + gamma * QB_tab[next_state][a_star] - old)
    else:
        a_star = max(QB_tab[next_state], key=QB_tab[next_state].get)  # pick with B
        old = QB_tab[state][action]
        QB_tab[state][action] = old + alpha * (reward + gamma * QA_tab[next_state][a_star] - old)


ready_r = True
try:
    for ep in range(REPLAY_EPISODES):
        ep_r = 0.0  # total rule reward this episode
        for i in range(len(ROWS) - 1):
            state = ROWS[i]["State"]
            cq = combined_q_r(state)
            # ε-greedy on the average of QA_r and QB_r
            action = RNG.choice(ACTIONS) if RNG.random() < epsilon else max(cq, key=cq.get)
            reward = REWARDS_DATA[(state, action)]
            next_state = ROWS[i + 1]["State"]
            ep_r += reward

            # <<< FILL: store this day — replay_buffer.append((state, action, reward, next_state))
            raise NotImplementedError("append the transition to replay_buffer")

            batch = sample_batch(replay_buffer, BATCH_SIZE)  # random past days
            for exp_s, exp_a, exp_r, exp_sp in batch:         # learn from the batch
                double_q_update_tables(QA_r, QB_r, exp_s, exp_a, exp_r, exp_sp, alpha, gamma)
        episode_returns.append(ep_r)
except Exception as err:
    ready_r = False
    print("NOT YET —", err)
    print("Fill sample_batch and the buffer.append line, then re-run.")

if ready_r:
    if len(replay_buffer) == 0:
        print("NOT YET — buffer is empty. Append (state, action, reward, next_state) each step.")
    else:
        policy_r = {s: max(combined_q_r(s), key=combined_q_r(s).get) for s in STATES}
        print("Replay policy:", policy_r)
        print("Buffer size:", len(replay_buffer), "| batch:", BATCH_SIZE)
        ok_r = (
            policy_r["Low Inventory"] == "Order"
            and policy_r["Medium Inventory"] == "Maintain"
            and policy_r["High Inventory"] == "Reduce"
        )
        print("CHECK 4c —", "PASS" if ok_r else "LOOK — expected Low→Order, Med→Maintain, High→Reduce")
        print("# === SEE (for REPORT 2) ===")
        print("Better result to report: same policy, but updates reuse shuffled past days (buffer), not only tomorrow.")
        if episode_returns:
            print("Episode return first → last: {:.1f} → {:.1f}".format(episode_returns[0], episode_returns[-1]))
        if plt is not None and episode_returns:
            fig, ax = plt.subplots(figsize=(5, 2.6))
            ax.plot(episode_returns, color="#2F5D50")
            ax.set_title("Episode return (rule rewards) · with replay")
            ax.set_xlabel("episode")
            fig.tight_layout()
            plt.show()


### Hint 4c (after you try)

```python
def sample_batch(buffer, batch_size):
    k = min(len(buffer), batch_size)
    if k == 0:
        return []
    return RNG.sample(list(buffer), k)

# inside the day loop, after computing next_state:
replay_buffer.append((state, action, reward, next_state))
```


### After CHECK 4c — what you should be able to say

**What is experience replay?** Store past `(s, a, r, s')` in a buffer; learn from a random mini-batch, not only from the next CSV row.

1. Updating only in file order **correlates** every step with the next day.
2. Replay **reuses** rare days and **breaks** that day-chain.
3. Tonight = **tables** + buffer. Session 9 = **network** + same buffer idea (DQN).

If the buffer stays empty, you forgot `append`. If CHECK 4c fails, check `sample_batch` and that Double Q still updates on the batch.


### Practice 2 REPORT · try → see → tell the room (⏱ end of 10 min)

After **CHECK 4c — PASS**, look at buffer size, policy, and the episode-return plot. Fill the report cell. Be ready to say what replay reused.


In [ ]:
# PRACTICE 2 REPORT — TRY THIS · experience replay

# Use numbers printed above (Stop 4c). Replace the blanks.

practice2_buffer_size = None       # <<< FILL: int, e.g. 2000
practice2_batch_size = None        # <<< FILL: int, e.g. 32
practice2_policy_still_good = None  # <<< FILL: True if CHECK 4c PASS
practice2_return_first = None      # <<< FILL: first episode_returns value (approx OK)
practice2_return_last = None       # <<< FILL: last episode_returns value (approx OK)
practice2_what_got_better = ""     # <<< FILL: one sentence (correlation / reuse / shuffle)

print("=== PRACTICE 2 REPORT ===")
print("buffer size:", practice2_buffer_size, "| batch:", practice2_batch_size)
print("CHECK 4c still good policy:", practice2_policy_still_good)
print("episode return first → last:", practice2_return_first, "→", practice2_return_last)
print("What got better:", practice2_what_got_better)

ok = (
    practice2_buffer_size is not None
    and practice2_batch_size is not None
    and practice2_policy_still_good is True
    and practice2_return_first is not None
    and practice2_return_last is not None
    and len(practice2_what_got_better.strip()) > 10
)
print("REPORT 2 —", "PASS (ready to share)" if ok else "NOT YET — fill every blank")
if ok and "episode_returns" in globals() and len(episode_returns) >= 2:
    print("(live check) first/last from run:", round(episode_returns[0], 1), "→", round(episode_returns[-1], 1))


## Stop 5 — Optional: Lab 4 Q3 (10 bins)

**CHANGE ME** `NUM_BINS` if you finish early.


In [ ]:
# STOP 5 — CHANGE ME · bins

NUM_BINS = 10  # <<< CHANGE ME: 5 or 15
inv_vals = [r["Current_Inventory"] for r in ROWS]
edges = np.linspace(min(inv_vals), max(inv_vals), num=NUM_BINS)


def to_bin(inv):
    return int(np.digitize(inv, edges))


def bin_reward(state, action):
    if state < 3 and action == "Order":
        return 2
    if 3 <= state < 7 and action == "Maintain":
        return 1
    if state >= 7 and action == "Reduce":
        return 2
    return -1


Qb = {s: {a: 0.0 for a in ACTIONS} for s in range(NUM_BINS + 1)}
for _ in range(30):
    for i in range(len(ROWS) - 1):
        s = to_bin(ROWS[i]["Current_Inventory"])
        a = RNG.choice(ACTIONS) if RNG.random() < 0.5 else max(Qb[s], key=Qb[s].get)
        sp = to_bin(ROWS[i + 1]["Current_Inventory"])
        old = Qb[s][a]
        Qb[s][a] = old + 0.3 * (bin_reward(s, a) + 0.8 * max(Qb[sp].values()) - old)

print("bins", NUM_BINS, "policy@0/mid/high:",
      max(Qb[0], key=Qb[0].get),
      max(Qb[NUM_BINS // 2], key=Qb[NUM_BINS // 2].get),
      max(Qb[NUM_BINS], key=Qb[NUM_BINS].get))


## Stop 6 — Diagnostics + deploy talk

| Knob | What breaks |
| --- | --- |
| `epsilon=0` | Freezes on a lucky first action |
| `alpha=1` | Chases the last sample |
| Bad rewards | Orders when High / reduces when Low |
| One table for pick + score | Overestimates lucky actions — try Double Q |
| Update only in day order | Correlated samples — try experience replay |

Would you put this Q-table on a real SKU tomorrow? Name the **human override** (Session 6 memo).


## Stop 7 — Framework (TRY THIS)


In [ ]:
# STOP 7 — TRY THIS

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL
print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
print("CHECK 7 —", "PASS" if framework_family and framework_line else "NOT YET")


## Raise a hand — Session 8 complete

**Timed blocks:** Practice 1 = Double Q (10 min). Practice 2 = experience replay (10 min).

1. [ ] Toy policy Low→Order, High→Reduce  
2. [ ] CHECK 4 PASS on Electronics days  
3. [ ] CHECK 4b PASS + **REPORT 1** shared (which Q more optimistic / what got better)  
4. [ ] CHECK 4c PASS + **REPORT 2** shared (buffer/batch + what replay reused)  
5. [ ] Changed α or ε and said what moved  
6. [ ] Named a human override for a bad Q row
